### Create a baseline to benchmark models on the Oct22 screening data


We create a held out test dataset of 15% of the data. 

In [1]:
import pathlib

import numpy as np
import pandas as pd

In [2]:
data_dir = pathlib.Path("../output")
data_csv_fn = data_dir / "ordinal_Oct22_sequences_with_dca_score.csv"

In [3]:
protein_letters = 'ACDEFGHIKLMNPQRSTVWY'
aa_map = {a:i for i, a in enumerate(protein_letters)}
q = len(protein_letters)

prot_to_list = lambda x: [aa_map[xi] for xi in x]

def one_hot_encode_list(l):
    int_seqs = np.array([prot_to_list(x) for x in l], dtype=int)
    return np.eye(q)[int_seqs].reshape(int_seqs.shape[0], -1)

In [4]:
class Oct22DataSet:
    
    """ Wrapper for the sequences dataset
    
        1. Read in the sequences dataset 
        2. Lightly preprocess
        3. Read in train/test splits (and cv splits if exist)
    """
    
    EXPECTED_NUM_CV_SPLITS = 5 # load upto these many cv splits
    
    def __init__(self, data_csv_fn):
        self.df = self._parse_df(data_csv_fn)
        self.L = len(self.df.sequence_aa_trim.iloc[0])
        self.train_indices = self._load_indices(data_csv_fn, "train")
        self.test_indices = self._load_indices(data_csv_fn, "test")
        
        # load in cv splits if they exist
        self.num_cv_splits = 0
        self.train_cv_splits = []
        self.val_cv_splits = []     
        self.load_cv_splits(data_csv_fn)
        
    def load_cv_splits(self, data_csv_fn):
        """ Load cv split indices. If no cv splits exist fail silently"""
        for i in range(self.EXPECTED_NUM_CV_SPLITS):
            try:
                train_idx_cv = self._load_indices(data_csv_fn, f"train_cv{i+1}")
                val_idx_cv = self._load_indices(data_csv_fn, f"val_cv{i+1}")
            except FileNotFoundError:
                print(f"File not found train_cv{i+1} or val_cv{i+1}. Ignoring...")
                pass
            else:
                self.train_cv_splits.append(train_idx_cv)
                self.val_cv_splits.append(val_idx_cv)
        self.num_cv_splits = len(self.train_cv_splits)
        
    def get_train_dataset(self):
        return self.df.iloc[self.train_indices]
    
    def get_test_dataset(self):
        return self.df.iloc[self.test_indices]
    
    def cv_iterator(self):
        """ Return training data split"""
        for i in range(self.num_cv_splits):
            yield (self.df.iloc[self.train_cv_splits[i]], #train cv split
                   self.df.iloc[self.val_cv_splits[i]]) # val cv split

    def get_n_splits(self, X=None, y=None, groups=None):
        """ For compatibility with sklearn CVsplitter"""
        return self.num_cv_splits
    
    def split(self, X, y=None, groups=None):
        """ 
            sklearn wants indices only so we cannot use self.cv_iterator()
            NOTE: sklearn wants indices of the training data
                  and not the indices of the dataset
            For compatibility with sklearn CVsplitter"""
        # make sure we are looking at training data
        assert(X.shape[0] == len(self.train_indices))
        for i in range(self.num_cv_splits):
            # convert the cv indices from self.df indices to train indices
            original_train_cv_indices = set(self.train_cv_splits[i])
            train_val_mask = np.array([x in original_train_cv_indices for x in 
                                       self.train_indices], dtype=bool)
            yield (np.where(train_val_mask)[0], np.where(~train_val_mask )[0])
        
        
    def __str__(self):
        return (f"aa_length :{self.L}\n" 
                f"num_seqs  :{len(self.df)}\n" 
                f"num_train :{len(self.train_indices)}\n"
                f"num_test  :{len(self.test_indices)}\n"
               )
        
    @staticmethod
    def _load_indices(data_csv_fn, idx_type):
        return np.loadtxt(data_csv_fn.with_suffix(f".{idx_type}.txt"), dtype=int)
        
    @staticmethod
    def _parse_df(data_csv_fn):
        df = pd.read_csv(data_csv_fn)
        df["library_num"] = df.parent.str.get(0).astype(int) - 1
        
        # map category to category code
        category_map = {'N':0, 'L':1, "P":2, "H":3}
        df["category_num"] = df["category"].map(category_map)
        
        # map activity to activity code (is_in_non_active_bin?)
        activity_map = {'N':0, 'L':1, "P":1, "H":1}
        df["activity_num"] = df["category"].map(activity_map)
        return df

In [5]:
ds = Oct22DataSet(data_csv_fn=data_csv_fn)
print(ds)

aa_length :272
num_seqs  :1326
num_train :1127
num_test  :199



In [6]:
def create_model_inputs(df, add_dca=False, ):
    """ Creates design matrix (adds dca or not)"""
    # add library num?
    design_matrix = one_hot_encode_list(df.sequence_aa_trim)
    if add_dca: # add dca to the last column
        design_matrix = np.hstack((design_matrix, df.dca_score.to_numpy()[:, np.newaxis]))
  
    return design_matrix


def create_target(df, activity_only=False):
    return df.activty_num.to_numpy() if activity_only \
        else df.category_num.to_numpy()

In [7]:
X = create_model_inputs(ds.get_train_dataset(), add_dca=False)
y = create_target(ds.get_train_dataset())

X.shape, y.shape

((1127, 5440), (1127,))

In [8]:
# for train, val in ds.cv_iterator():
#     X = create_model_inputs(train, add_dca=True)
#     y = create_target(train)
#     print(X.shape, y.shape)

In [9]:
# for i, (train_index, val_index) in enumerate(ds.split(ds.get_train_dataset())):
#     print(f"Fold {i}:")
#     print(f"  Train: index={train_index}")
#     print(f"  Test:  index={val_index}")

In [10]:
import sklearn
from sklearn.linear_model import RidgeClassifierCV



model = sklearn.linear_model.RidgeClassifierCV(fit_intercept=False, cv=ds)

In [11]:
model.fit(X, y)

RidgeClassifierCV(cv=<__main__.Oct22DataSet object at 0x7f80344af700>,
                  fit_intercept=False)

In [12]:
model.score(X, y)

0.7914818101153505